# Simulación de impacto económico

En este notebook simulo el impacto económico potencial de reducir el sobrestock identificado en el análisis exploratorio.

Mi objetivo es estimar cuánto capital podría liberarse, cuánto ahorro operativo anual podría generarse y qué escenario sería más razonable para una implementación inicial.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "reports"

ventas = pd.read_csv(DATA_DIR / "fact_ventas.csv", parse_dates=["fecha", "fecha_mes"])
inventario = pd.read_csv(DATA_DIR / "fact_inventario.csv", parse_dates=["fecha_corte", "fecha_mes"])
productos = pd.read_csv(DATA_DIR / "dim_producto.csv")
oportunidades = pd.read_csv(REPORTS_DIR / "oportunidades_priorizadas.csv")

print("Ventas:", ventas.shape)
print("Inventario:", inventario.shape)
print("Oportunidades:", oportunidades.shape)

Ventas: (46870, 14)
Inventario: (5760, 11)
Oportunidades: (15, 10)


In [2]:
supuestos = pd.DataFrame({
    "supuesto": [
        "Costo anual de mantener inventario",
        "Reducción conservadora de sobrestock",
        "Reducción moderada de sobrestock",
        "Reducción agresiva de sobrestock",
        "Horizonte de evaluación"
    ],
    "valor": [
        "12%",
        "25%",
        "40%",
        "60%",
        "12 meses"
    ],
    "comentario": [
        "Supuesto usado para estimar ahorro operativo por menor almacenamiento, seguros, manipulación y costo financiero.",
        "Escenario de baja intervención y menor riesgo operativo.",
        "Escenario recomendado como meta inicial por equilibrio entre impacto y riesgo.",
        "Escenario de intervención fuerte, requiere mayor control de demanda y reposición.",
        "Periodo usado para estimar impacto anual."
    ]
})

supuestos

,supuesto,valor,comentario
0,Costo anual de mantener inventario,12%,Supuesto usado para estimar ahorro operativo p...
1,Reducción conservadora de sobrestock,25%,Escenario de baja intervención y menor riesgo ...
2,Reducción moderada de sobrestock,40%,Escenario recomendado como meta inicial por eq...
3,Reducción agresiva de sobrestock,60%,"Escenario de intervención fuerte, requiere may..."
4,Horizonte de evaluación,12 meses,Periodo usado para estimar impacto anual.


In [3]:
ventas_netas_total = ventas["ingreso_neto"].sum()
sobrestock_total = inventario["sobrestock_valorizado"].sum()
costo_mantenimiento_anual_pct = 0.12

ventas_netas_total, sobrestock_total

(np.float64(925593305.9200001), np.float64(34381294.620000005))

In [4]:
escenarios = pd.DataFrame({
    "escenario": ["Conservador", "Moderado", "Agresivo"],
    "reduccion_sobrestock_pct": [0.25, 0.40, 0.60]
})

escenarios["capital_liberado"] = (
    sobrestock_total * escenarios["reduccion_sobrestock_pct"]
)

escenarios["ahorro_operativo_anual"] = (
    escenarios["capital_liberado"] * costo_mantenimiento_anual_pct
)

escenarios["impacto_total_referencial"] = (
    escenarios["capital_liberado"] + escenarios["ahorro_operativo_anual"]
)

escenarios["capital_liberado_sobre_ventas_pct"] = (
    escenarios["capital_liberado"] / ventas_netas_total
)

escenarios["ahorro_operativo_sobre_ventas_pct"] = (
    escenarios["ahorro_operativo_anual"] / ventas_netas_total
)

escenarios

,escenario,reduccion_sobrestock_pct,capital_liberado,ahorro_operativo_anual,impacto_total_referencial,capital_liberado_sobre_ventas_pct,ahorro_operativo_sobre_ventas_pct
0,Conservador,0.25,"8,595,323.66","1,031,438.84","9,626,762.49",0.01,0.00
1,Moderado,0.40,"13,752,517.85","1,650,302.14","15,402,819.99",0.01,0.00
2,Agresivo,0.60,"20,628,776.77","2,475,453.21","23,104,229.98",0.02,0.00


In [5]:
escenarios_formateados = escenarios.copy()

for col in [
    "reduccion_sobrestock_pct",
    "capital_liberado_sobre_ventas_pct",
    "ahorro_operativo_sobre_ventas_pct"
]:
    escenarios_formateados[col] = escenarios_formateados[col] * 100

escenarios_formateados

,escenario,reduccion_sobrestock_pct,capital_liberado,ahorro_operativo_anual,impacto_total_referencial,capital_liberado_sobre_ventas_pct,ahorro_operativo_sobre_ventas_pct
0,Conservador,25.00,"8,595,323.66","1,031,438.84","9,626,762.49",0.93,0.11
1,Moderado,40.00,"13,752,517.85","1,650,302.14","15,402,819.99",1.49,0.18
2,Agresivo,60.00,"20,628,776.77","2,475,453.21","23,104,229.98",2.23,0.27


In [6]:
impacto_producto = oportunidades.copy()

impacto_producto["capital_liberado_moderado"] = (
    impacto_producto["sobrestock_valorizado"] * 0.40
)

impacto_producto["ahorro_operativo_anual_moderado"] = (
    impacto_producto["capital_liberado_moderado"] * costo_mantenimiento_anual_pct
)

impacto_producto["impacto_total_referencial_moderado"] = (
    impacto_producto["capital_liberado_moderado"] 
    + impacto_producto["ahorro_operativo_anual_moderado"]
)

impacto_producto = impacto_producto.sort_values(
    "impacto_total_referencial_moderado",
    ascending=False
)

impacto_producto[
    [
        "producto",
        "categoria",
        "estado_catalogo",
        "margen_pct",
        "sobrestock_valorizado",
        "capital_liberado_moderado",
        "ahorro_operativo_anual_moderado",
        "impacto_total_referencial_moderado",
        "accion_sugerida"
    ]
]

,producto,categoria,estado_catalogo,margen_pct,sobrestock_valorizado,capital_liberado_moderado,ahorro_operativo_anual_moderado,impacto_total_referencial_moderado,accion_sugerida
0,Cemento Portland Tipo I Granel,Cemento granel,Activo,0.26,"6,340,462.83","2,536,185.13","304,342.22","2,840,527.35",Ajustar planificación sin afectar disponibilidad
1,Cemento Tipo V Granel,Cemento granel,Observar,0.23,"5,712,831.84","2,285,132.74","274,215.93","2,559,348.66",Reducir compras y monitorear demanda
2,Cemento Alta Resistencia Granel,Cemento granel,Activo,0.25,"2,658,787.80","1,063,515.12","127,621.81","1,191,136.93",Mantener seguimiento
3,Cemento Premium Acabados 42.5kg,Cemento embolsado,Observar,0.23,"2,123,327.38","849,330.95","101,919.71","951,250.67",Reducir compras y monitorear demanda
4,Concreto autocompactante,Concreto premezclado,Observar,0.17,"2,118,622.25","847,448.90","101,693.87","949,142.77","Revisar precio, costo y política de reposición"
5,Cemento blanco decorativo,Cemento embolsado,Descontinuar,0.13,"2,100,882.96","840,353.18","100,842.38","941,195.57",Liquidar inventario y retirar del catálogo
6,Cemento Portland IP Granel,Cemento granel,Activo,0.26,"2,029,643.30","811,857.32","97,422.88","909,280.20",Ajustar planificación sin afectar disponibilidad
7,Cemento Economico 42.5kg,Cemento embolsado,Observar,0.15,"1,959,803.47","783,921.39","94,070.57","877,991.95","Revisar precio, costo y política de reposición"
8,Cemento Antisalitre 42.5kg,Cemento embolsado,Activo,0.26,"1,924,704.30","769,881.72","92,385.81","862,267.53",Ajustar planificación sin afectar disponibilidad
9,Cemento Alta Resistencia 42.5kg,Cemento embolsado,Activo,0.26,"1,228,220.99","491,288.40","58,954.61","550,243.00",Ajustar planificación sin afectar disponibilidad


In [7]:
impacto_accion = (
    impacto_producto
    .groupby("accion_sugerida", as_index=False)
    .agg(
        productos=("producto", "count"),
        sobrestock_valorizado=("sobrestock_valorizado", "sum"),
        capital_liberado_moderado=("capital_liberado_moderado", "sum"),
        ahorro_operativo_anual_moderado=("ahorro_operativo_anual_moderado", "sum"),
        impacto_total_referencial_moderado=("impacto_total_referencial_moderado", "sum")
    )
    .sort_values("impacto_total_referencial_moderado", ascending=False)
)

impacto_accion

,accion_sugerida,productos,sobrestock_valorizado,capital_liberado_moderado,ahorro_operativo_anual_moderado,impacto_total_referencial_moderado
0,Ajustar planificación sin afectar disponibilidad,5,"11,970,800.98","4,788,320.39","574,598.45","5,362,918.84"
3,Reducir compras y monitorear demanda,2,"7,836,159.22","3,134,463.69","376,135.64","3,510,599.33"
2,Mantener seguimiento,5,"5,317,620.58","2,127,048.23","255,245.79","2,382,294.02"
4,"Revisar precio, costo y política de reposición",2,"4,078,425.72","1,631,370.29","195,764.43","1,827,134.72"
1,Liquidar inventario y retirar del catálogo,1,"2,100,882.96","840,353.18","100,842.38","941,195.57"


## Interpretación de resultados

### Hallazgo 9: el escenario moderado es el más defendible

Comparé tres escenarios de reducción de sobrestock: conservador, moderado y agresivo. El escenario moderado, con una reducción del 40%, permite liberar aproximadamente S/ 13.75M de capital inmovilizado y generar un ahorro operativo anual estimado de S/ 1.65M.

Elegí este escenario como referencia porque equilibra impacto económico y riesgo operativo. No busca reducir inventario de forma extrema, sino optimizarlo sin afectar la disponibilidad de productos clave.

### Hallazgo 10: la mayor oportunidad está en ajustar planificación, no solo liquidar productos

Al agrupar el impacto por acción sugerida, observé que la mayor parte del impacto referencial moderado está en la acción "Ajustar planificación sin afectar disponibilidad", con S/ 5.36M. Esto confirma que el problema no se resuelve únicamente liquidando productos, sino mejorando la planificación de compras, inventario y demanda.

También encontré oportunidades relevantes en reducir compras y monitorear demanda, especialmente para productos en estado de observación.

### Hallazgo 11: los productos core requieren una estrategia distinta

Cemento Portland Tipo I Granel concentra el mayor impacto potencial individual, con S/ 2.84M en el escenario moderado. Sin embargo, al ser un producto activo y relevante para el negocio, no debería gestionarse como producto de baja prioridad.

En este tipo de producto, la recomendación es ajustar la planificación y la reposición sin afectar disponibilidad. En cambio, productos como Cemento blanco decorativo, Concreto autocompactante o Cemento Económico 42.5kg requieren revisión de catálogo, precio, costo y política de reposición.

In [8]:
REPORTS_DIR.mkdir(exist_ok=True)

supuestos.to_csv(
    REPORTS_DIR / "supuestos_simulacion_impacto.csv",
    index=False,
    encoding="utf-8-sig"
)

escenarios.to_csv(
    REPORTS_DIR / "escenarios_impacto_economico.csv",
    index=False,
    encoding="utf-8-sig"
)

impacto_producto.to_csv(
    REPORTS_DIR / "impacto_producto_moderado.csv",
    index=False,
    encoding="utf-8-sig"
)

impacto_accion.to_csv(
    REPORTS_DIR / "impacto_accion_moderado.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Resultados de simulación exportados correctamente.")

Resultados de simulación exportados correctamente.


In [9]:
escenario_recomendado = escenarios.loc[
    escenarios["escenario"] == "Moderado"
].iloc[0]

resumen_simulacion = pd.DataFrame({
    "indicador": [
        "Escenario recomendado",
        "Reducción de sobrestock",
        "Capital liberado",
        "Ahorro operativo anual",
        "Impacto total referencial",
        "Capital liberado sobre ventas"
    ],
    "valor": [
        escenario_recomendado["escenario"],
        f'{escenario_recomendado["reduccion_sobrestock_pct"]:.0%}',
        f'S/ {escenario_recomendado["capital_liberado"]:,.2f}',
        f'S/ {escenario_recomendado["ahorro_operativo_anual"]:,.2f}',
        f'S/ {escenario_recomendado["impacto_total_referencial"]:,.2f}',
        f'{escenario_recomendado["capital_liberado_sobre_ventas_pct"]:.2%}'
    ]
})

resumen_simulacion

,indicador,valor
0,Escenario recomendado,Moderado
1,Reducción de sobrestock,40%
2,Capital liberado,"S/ 13,752,517.85"
3,Ahorro operativo anual,"S/ 1,650,302.14"
4,Impacto total referencial,"S/ 15,402,819.99"
5,Capital liberado sobre ventas,1.49%
